# Notebook 09 — Coupled Drift MPC vs Kalman

Coupled-drift control experiment for `quantum-hardware-company/control-stack`.

This notebook extends Notebook 08 by adding **coupled Ω + B drift**, then compares:

- no control
- moving average
- scalar Kalman
- joint Kalman
- naive predictive control
- coupled MPC
- oracle reference

Outputs are saved using the repo convention:

`figures/coupled_drift_mpc/09_coupled_drift_mpc_<name>.png`

and

`results/coupled_drift_mpc_summary.json`

In [ ]:
import os
import json
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(9423)

FIG_DIR = Path("figures/coupled_drift_mpc")
RESULTS_DIR = Path("results")
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PREFIX = "09_coupled_drift_mpc"

def save_fig(name):
    """Save figures in repo-standard Notebook 09 format."""
    base = FIG_DIR / f"{NOTEBOOK_PREFIX}_{name}"
    plt.savefig(f"{base}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{base}.pdf", bbox_inches="tight")

print("Output directories ready:")
print(" -", FIG_DIR)
print(" -", RESULTS_DIR)

## 1. Synthetic coupled-drift calibration stream

We create a calibration stream where Ω and B are not independent:

- Ω has fast oscillatory drift.
- B has slow drift plus weak Ω-coupled modulation.
- Measurements include independent readout noise.

This creates a stronger control test than the previous notebooks.

In [ ]:
T = 48
blocks = np.arange(T)
t = blocks.astype(float)

# True coupled drift structure.
true_omega = (
    0.070 * np.sin(0.34 * t + 0.2)
    + 0.028 * np.sin(0.83 * t - 0.4)
    + 0.010 * np.sin(1.37 * t)
)

true_B_base = 0.006 + 0.00125 * t + 0.010 * np.sin(0.22 * t + 0.6)
true_B = true_B_base + 0.10 * true_omega + 0.006 * np.sin(0.51 * t)

# Measurement noise.
sigma_omega = 0.0035
sigma_B = 0.0060
measured_omega = true_omega + np.random.normal(0, sigma_omega, size=T)
measured_B = true_B + np.random.normal(0, sigma_B, size=T)

print(f"blocks: {T}")
print(f"true Ω/B drift correlation: {np.corrcoef(true_omega, true_B)[0,1]:.4f}")
print(f"measured Ω/B drift correlation: {np.corrcoef(measured_omega, measured_B)[0,1]:.4f}")

## 2. Response model and metrics

The notebook uses a simple Rabi-like response model. The target response uses true drift. Each policy produces corrected Ω and B estimates, then generates a response compared to target.

In [ ]:
x_axis = np.linspace(0, 10, 240)

OMEGA0 = 2.47
B0 = 0.060

def response_curve(omega_drift, b_drift, x=x_axis):
    """Rabi-like response curve with frequency and offset drift."""
    omega = OMEGA0 + omega_drift
    b = B0 + b_drift
    y = b + 0.47 * (1.0 + np.sin(omega * x + 0.15))
    return np.clip(y, 0, 1.05)

def build_response(omega_series, b_series):
    return np.array([response_curve(o, b) for o, b in zip(omega_series, b_series)])

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def cosine_similarity(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return float("nan")
    return float(np.dot(a, b) / denom)

target_response = build_response(true_omega, true_B)
threshold = 1 / np.sqrt(1**2 + 1**2)
print("CGCS threshold:", threshold)

## 3. Estimators and control policies

In [ ]:
def moving_average(z, window=4):
    z = np.asarray(z)
    out = np.zeros_like(z, dtype=float)
    for i in range(len(z)):
        lo = max(0, i - window + 1)
        out[i] = np.mean(z[lo:i+1])
    return out


def scalar_kalman(z, q=8e-4, r=None):
    z = np.asarray(z, dtype=float)
    if r is None:
        r = np.var(z[:6] - np.mean(z[:6])) + 1e-6
    x = z[0]
    P = 1.0
    xs = []
    gains = []
    for zi in z:
        P = P + q
        K = P / (P + r)
        x = x + K * (zi - x)
        P = (1 - K) * P
        xs.append(x)
        gains.append(K)
    return np.array(xs), np.array(gains)


def joint_kalman(z_omega, z_B, q_omega=8e-4, q_B=1e-5, coupling=0.25, r_omega=None, r_B=None):
    z_omega = np.asarray(z_omega, dtype=float)
    z_B = np.asarray(z_B, dtype=float)
    if r_omega is None:
        r_omega = sigma_omega**2
    if r_B is None:
        r_B = sigma_B**2

    q_cross = coupling * np.sqrt(q_omega * q_B)
    Q = np.array([[q_omega, q_cross], [q_cross, q_B]])
    R = np.array([[r_omega, 0.0], [0.0, r_B]])
    H = np.eye(2)

    x = np.array([z_omega[0], z_B[0]], dtype=float)
    P = np.eye(2) * 0.05
    xs = []
    covars = []
    gains = []

    for zo, zb in zip(z_omega, z_B):
        # predict
        P = P + Q
        z = np.array([zo, zb])

        # update
        S = H @ P @ H.T + R
        K = P @ H.T @ np.linalg.inv(S)
        x = x + K @ (z - H @ x)
        P = (np.eye(2) - K @ H) @ P

        xs.append(x.copy())
        covars.append(P.copy())
        gains.append(K.copy())

    return np.array(xs), np.array(covars), np.array(gains), Q, R


def coupled_mpc_command(joint_est, alpha=0.82, command_bound_omega=0.030, command_bound_B=0.014):
    """MPC-lite: smooth joint estimate and clip command updates."""
    cmd = np.zeros_like(joint_est)
    cmd[0] = joint_est[0]
    for i in range(1, len(joint_est)):
        proposed = alpha * joint_est[i] + (1 - alpha) * cmd[i-1]
        delta = proposed - cmd[i-1]
        delta[0] = np.clip(delta[0], -command_bound_omega, command_bound_omega)
        delta[1] = np.clip(delta[1], -command_bound_B, command_bound_B)
        cmd[i] = cmd[i-1] + delta
    return cmd

# Policies.
ma_omega = moving_average(measured_omega, window=4)
ma_B = moving_average(measured_B, window=4)

sk_omega, gain_omega = scalar_kalman(measured_omega, q=8e-4, r=sigma_omega**2)
sk_B, gain_B = scalar_kalman(measured_B, q=1e-5, r=sigma_B**2)

jk, jk_covars, jk_gains, Q, R = joint_kalman(measured_omega, measured_B, coupling=0.25)
jk_omega = jk[:, 0]
jk_B = jk[:, 1]

# Naive predictive baseline: extrapolate one-step from measured drift.
naive_omega = np.copy(measured_omega)
naive_B = np.copy(measured_B)
for i in range(2, T):
    naive_omega[i] = measured_omega[i-1] + (measured_omega[i-1] - measured_omega[i-2])
    naive_B[i] = measured_B[i-1] + (measured_B[i-1] - measured_B[i-2])

mpc_state = coupled_mpc_command(jk, alpha=0.82, command_bound_omega=0.030, command_bound_B=0.014)
mpc_omega = mpc_state[:, 0]
mpc_B = mpc_state[:, 1]

oracle_omega = true_omega.copy()
oracle_B = true_B.copy()

policies = {
    "none": (measured_omega, measured_B),
    "moving_average": (ma_omega, ma_B),
    "scalar_kalman": (sk_omega, sk_B),
    "joint_kalman": (jk_omega, jk_B),
    "naive_predictive": (naive_omega, naive_B),
    "coupled_mpc": (mpc_omega, mpc_B),
    "oracle": (oracle_omega, oracle_B),
}

responses = {name: build_response(o, b) for name, (o, b) in policies.items()}

print("Q =")
print(Q)
print("R =")
print(R)

## 4. Summary metrics

In [ ]:
summary = {}
for name, resp in responses.items():
    per_block_rmse = np.array([rmse(resp[i], target_response[i]) for i in range(T)])
    per_block_cos = np.array([cosine_similarity(resp[i], target_response[i]) for i in range(T)])
    omega_err = np.asarray(policies[name][0]) - true_omega
    b_err = np.asarray(policies[name][1]) - true_B
    summary[name] = {
        "mean_response_rmse": float(np.mean(per_block_rmse)),
        "max_response_rmse": float(np.max(per_block_rmse)),
        "omega_rmse": rmse(policies[name][0], true_omega),
        "offset_rmse": rmse(policies[name][1], true_B),
        "min_cosine": float(np.min(per_block_cos)),
        "mean_cosine": float(np.mean(per_block_cos)),
        "max_angle_deg": float(np.degrees(np.arccos(np.clip(np.min(per_block_cos), -1, 1))))
    }

ranking = sorted(summary.keys(), key=lambda k: summary[k]["mean_response_rmse"])
print("Policy ranking by mean response RMSE:")
for i, name in enumerate(ranking, 1):
    print(f"{i:>2}. {name:<17} mean={summary[name]['mean_response_rmse']:.5f} max={summary[name]['max_response_rmse']:.5f}")

## 5. Ω tracking and control command

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(blocks, true_omega, label="true Ω drift", linewidth=2)
plt.plot(blocks, measured_omega, marker="o", alpha=0.65, label="measured Ω drift")
plt.plot(blocks, ma_omega, linestyle="--", label="moving average")
plt.plot(blocks, sk_omega, linestyle=":", linewidth=2.5, label="scalar Kalman")
plt.plot(blocks, jk_omega, linestyle="-.", linewidth=2.2, label="joint Kalman")
plt.plot(blocks, mpc_omega, linewidth=2.2, label="coupled MPC")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Ω drift estimate / command")
plt.title("Coupled drift: Ω tracking and control command")
plt.legend()
save_fig("omega_tracking")
plt.show()

## 6. B tracking and control command

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(blocks, true_B, label="true B drift", linewidth=2)
plt.plot(blocks, measured_B, marker="o", alpha=0.65, label="measured B drift")
plt.plot(blocks, ma_B, linestyle="--", label="moving average")
plt.plot(blocks, sk_B, linestyle=":", linewidth=2.5, label="scalar Kalman")
plt.plot(blocks, jk_B, linestyle="-.", linewidth=2.2, label="joint Kalman")
plt.plot(blocks, mpc_B, linewidth=2.2, label="coupled MPC")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("B drift estimate / command")
plt.title("Coupled drift: B tracking and control command")
plt.legend()
save_fig("offset_tracking")
plt.show()

## 7. Response-level error comparison

In [ ]:
plt.figure(figsize=(10, 5))
for name, resp in responses.items():
    errs = [rmse(resp[i], target_response[i]) for i in range(T)]
    plt.plot(blocks, errs, marker="o", label=name)
plt.xlabel("calibration block / lab time index")
plt.ylabel("RMSE vs target response")
plt.title("Coupled drift: response-level error comparison")
plt.legend()
save_fig("response_rmse_comparison")
plt.show()

## 8. Policy ranking

In [ ]:
rank_names = ranking
mean_vals = [summary[k]["mean_response_rmse"] for k in rank_names]
max_vals = [summary[k]["max_response_rmse"] for k in rank_names]

x = np.arange(len(rank_names))
w = 0.36
plt.figure(figsize=(10, 5))
plt.bar(x - w/2, mean_vals, width=w, label="mean RMSE")
plt.bar(x + w/2, max_vals, width=w, label="max RMSE")
for xi, val in zip(x - w/2, mean_vals):
    plt.text(xi, val, f"{val:.3f}", ha="center", va="bottom", fontsize=9)
for xi, val in zip(x + w/2, max_vals):
    plt.text(xi, val, f"{val:.3f}", ha="center", va="bottom", fontsize=9)
plt.xticks(x, rank_names, rotation=25, ha="right")
plt.ylabel("response RMSE")
plt.title("Coupled drift: control policy ranking")
plt.legend()
save_fig("policy_ranking_summary")
plt.show()

## 9. CGCS phase-lock stability

In [ ]:
plt.figure(figsize=(10, 5))
for name, resp in responses.items():
    cos_vals = [cosine_similarity(resp[i], target_response[i]) for i in range(T)]
    plt.plot(blocks, cos_vals, marker="o", label=name)
plt.axhline(threshold, linestyle="--", label="45° threshold")
plt.xlabel("calibration block / lab time index")
plt.ylabel("cosine similarity vs target")
plt.title("Coupled drift: CGCS phase-lock stability")
plt.legend(loc="lower right")
save_fig("cgcs_stability")
plt.show()

## 10. Worst-case block comparison

In [ ]:
# Worst case for best practical policy, excluding oracle.
best_practical = [k for k in ranking if k != "oracle"][0]
worst_idx = int(np.argmax([rmse(responses[best_practical][i], target_response[i]) for i in range(T)]))

plt.figure(figsize=(10, 5))
plt.plot(x_axis, target_response[worst_idx], linewidth=2.8, label="target")
for name in ["none", "moving_average", "scalar_kalman", "joint_kalman", "naive_predictive", "coupled_mpc", "oracle"]:
    plt.plot(x_axis, responses[name][worst_idx], label=name)
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Coupled drift: worst-case block {worst_idx}")
plt.legend()
save_fig("worst_case_block_comparison")
plt.show()

print("Worst-case block for best practical policy:", worst_idx)
print("Best practical policy:", best_practical)

## 11. Coupling sweep

Sweep joint-process covariance coupling to test whether a coupled estimator helps response-level performance.

In [ ]:
coupling_grid = np.linspace(-0.75, 0.75, 31)
sweep_rmse = []
for c in coupling_grid:
    jk_sweep, *_ = joint_kalman(measured_omega, measured_B, coupling=float(c))
    resp = build_response(jk_sweep[:, 0], jk_sweep[:, 1])
    sweep_rmse.append(np.mean([rmse(resp[i], target_response[i]) for i in range(T)]))

sweep_rmse = np.array(sweep_rmse)
best_coupling = float(coupling_grid[np.argmin(sweep_rmse)])
current_coupling = 0.25

plt.figure(figsize=(10, 5))
plt.plot(coupling_grid, sweep_rmse, marker="o")
plt.axvline(current_coupling, linestyle="--", label=f"current coupling = {current_coupling:.2f}")
plt.axvline(best_coupling, linestyle=":", label=f"best coupling = {best_coupling:.2f}")
plt.xlabel("process covariance coupling coefficient")
plt.ylabel("mean response RMSE")
plt.title("Coupled drift: joint Kalman coupling sweep")
plt.legend()
save_fig("coupling_sweep")
plt.show()

print("best coupling:", best_coupling)

## 12. Command-bound sweep

Sweep MPC command bounds to expose the tradeoff between smoothness and tracking lag.

In [ ]:
bounds = np.array([0.004, 0.007, 0.010, 0.014, 0.020, 0.030, 0.045])
bound_rmse = []
for bnd in bounds:
    cmd = coupled_mpc_command(jk, alpha=0.82, command_bound_omega=float(bnd), command_bound_B=float(bnd / 2.2))
    resp = build_response(cmd[:, 0], cmd[:, 1])
    bound_rmse.append(np.mean([rmse(resp[i], target_response[i]) for i in range(T)]))

bound_rmse = np.array(bound_rmse)
best_bound = float(bounds[np.argmin(bound_rmse)])
current_bound = 0.030

plt.figure(figsize=(10, 5))
plt.plot(bounds, bound_rmse, marker="o")
plt.axvline(current_bound, linestyle="--", label=f"current Ω bound = {current_bound:.3f}")
plt.axvline(best_bound, linestyle=":", label=f"best Ω bound = {best_bound:.3f}")
plt.xlabel("allowed ΔΩ command radius")
plt.ylabel("mean response RMSE")
plt.title("Coupled drift: command-bound sweep")
plt.legend()
save_fig("command_bound_sweep")
plt.show()

print("best command bound:", best_bound)

## 13. Save results summary and downloadable ZIP

In [ ]:
result_payload = {
    "notebook": "09_coupled_drift_mpc",
    "blocks": int(T),
    "phase_lock_threshold": float(threshold),
    "true_omega_b_correlation": float(np.corrcoef(true_omega, true_B)[0, 1]),
    "measured_omega_b_correlation": float(np.corrcoef(measured_omega, measured_B)[0, 1]),
    "sigma_omega": float(sigma_omega),
    "sigma_B": float(sigma_B),
    "joint_kalman": {
        "q_omega": float(Q[0, 0]),
        "q_B": float(Q[1, 1]),
        "q_cross": float(Q[0, 1]),
        "current_coupling": float(current_coupling),
        "best_coupling": float(best_coupling),
    },
    "mpc": {
        "alpha": 0.82,
        "current_omega_bound": float(current_bound),
        "best_omega_bound": float(best_bound),
    },
    "ranking": ranking,
    "summary": summary,
}

summary_path = RESULTS_DIR / "coupled_drift_mpc_summary.json"
with open(summary_path, "w") as f:
    json.dump(result_payload, f, indent=2)

zip_path = "coupled_drift_mpc_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in FIG_DIR.glob("09_coupled_drift_mpc_*"):
        z.write(path, arcname=str(path))
    z.write(summary_path, arcname=str(summary_path))

print("Saved:", summary_path)
print("Saved:", zip_path)
print("Figures:")
for p in sorted(FIG_DIR.glob("09_coupled_drift_mpc_*.png")):
    print(" -", p)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Not running in Colab; ZIP is available locally at:", zip_path)